## LLM API Configuration

The LLM module uses two providers, Google Gemini and OpenRouter.

Both providers receive the same XGBoost prediction and SHAP-based evidence so that their generated explanations can be compared fairly.

API keys are stored securely in environment variables and are not directly written in the notebook.

In [20]:
# Load SHAP evidance 

import json

with open("../models/shap_llm_evidence.json", "r") as f:
    llm_evidence = json.load(f)

print("SHAP evidence loaded successfully!")
print(llm_evidence)

SHAP evidence loaded successfully!
{'prediction': 'Phishing', 'phishing_probability': 0.9999977350234985, 'top_features': [{'Feature': 'URLSimilarityIndex', 'SHAP Value': 9.221220970153809, 'Feature Value': 100.0}, {'Feature': 'NoOfSelfRef', 'SHAP Value': 0.81537926197052, 'Feature Value': 119.0}, {'Feature': 'NoOfImage', 'SHAP Value': 0.6637913584709167, 'Feature Value': 34.0}, {'Feature': 'LineOfCode', 'SHAP Value': 0.4491221010684967, 'Feature Value': 558.0}, {'Feature': 'NoOfJS', 'SHAP Value': 0.3792590796947479, 'Feature Value': 28.0}, {'Feature': 'NoOfCSS', 'SHAP Value': 0.2874261736869812, 'Feature Value': 20.0}, {'Feature': 'URLCharProb', 'SHAP Value': 0.2782759368419647, 'Feature Value': 0.061933179}, {'Feature': 'LetterRatioInURL', 'SHAP Value': -0.22271820902824402, 'Feature Value': 0.581}, {'Feature': 'SpacialCharRatioInURL', 'SHAP Value': 0.1979997754096985, 'Feature Value': 0.032}, {'Feature': 'NoOfExternalRef', 'SHAP Value': 0.19639922678470612, 'Feature Value': 124.0}]}

## Inspect SHAP Evidence

The saved SHAP evidence contains the XGBoost prediction, phishing probability, and the top contributing features.

This information will be used to construct the LLM prompt.

In [21]:
print("Prediction:", llm_evidence["prediction"])
print(
    "Phishing Probability:",
    f"{llm_evidence['phishing_probability']:.2%}"
)

print("\nTop SHAP Features:")

for feature in llm_evidence["top_features"]:
    print(
        f"{feature['Feature']}: "
        f"SHAP={feature['SHAP Value']:.4f}, "
        f"Value={feature['Feature Value']}"
    )

Prediction: Phishing
Phishing Probability: 100.00%

Top SHAP Features:
URLSimilarityIndex: SHAP=9.2212, Value=100.0
NoOfSelfRef: SHAP=0.8154, Value=119.0
NoOfImage: SHAP=0.6638, Value=34.0
LineOfCode: SHAP=0.4491, Value=558.0
NoOfJS: SHAP=0.3793, Value=28.0
NoOfCSS: SHAP=0.2874, Value=20.0
URLCharProb: SHAP=0.2783, Value=0.061933179
LetterRatioInURL: SHAP=-0.2227, Value=0.581
SpacialCharRatioInURL: SHAP=0.1980, Value=0.032
NoOfExternalRef: SHAP=0.1964, Value=124.0


In [22]:
shap_evidence = "\n".join(
    [
        f"- {feature['Feature']}: "
        f"SHAP contribution = {feature['SHAP Value']:.4f}, "
        f"Feature value = {feature['Feature Value']}"
        for feature in llm_evidence["top_features"]
    ]
)

print(shap_evidence)

- URLSimilarityIndex: SHAP contribution = 9.2212, Feature value = 100.0
- NoOfSelfRef: SHAP contribution = 0.8154, Feature value = 119.0
- NoOfImage: SHAP contribution = 0.6638, Feature value = 34.0
- LineOfCode: SHAP contribution = 0.4491, Feature value = 558.0
- NoOfJS: SHAP contribution = 0.3793, Feature value = 28.0
- NoOfCSS: SHAP contribution = 0.2874, Feature value = 20.0
- URLCharProb: SHAP contribution = 0.2783, Feature value = 0.061933179
- LetterRatioInURL: SHAP contribution = -0.2227, Feature value = 0.581
- SpacialCharRatioInURL: SHAP contribution = 0.1980, Feature value = 0.032
- NoOfExternalRef: SHAP contribution = 0.1964, Feature value = 124.0


## LLM Prompt Construction

A structured prompt is created using the XGBoost prediction, phishing probability, and SHAP feature contributions.

The LLM is instructed to explain the existing model result rather than making a new prediction.

In [23]:
prediction = llm_evidence["prediction"]
probability = llm_evidence["phishing_probability"]

prompt = f"""
You are an AI cybersecurity explanation assistant for PhishGuard AI.

The machine learning model has already classified the URL.

Prediction: {prediction}
Phishing Probability: {probability:.2%}

The following SHAP features contributed to this prediction:

{shap_evidence}

Explain the model's decision in simple and clear language.

Requirements:
- Do not make a new prediction.
- Do not change the provided prediction.
- Use the SHAP evidence as the basis of the explanation.
- Explain the most important features.
- Avoid unnecessary technical terminology.
- Clearly state that the explanation is based on the machine learning model.
"""

print(prompt)


You are an AI cybersecurity explanation assistant for PhishGuard AI.

The machine learning model has already classified the URL.

Prediction: Phishing
Phishing Probability: 100.00%

The following SHAP features contributed to this prediction:

- URLSimilarityIndex: SHAP contribution = 9.2212, Feature value = 100.0
- NoOfSelfRef: SHAP contribution = 0.8154, Feature value = 119.0
- NoOfImage: SHAP contribution = 0.6638, Feature value = 34.0
- LineOfCode: SHAP contribution = 0.4491, Feature value = 558.0
- NoOfJS: SHAP contribution = 0.3793, Feature value = 28.0
- NoOfCSS: SHAP contribution = 0.2874, Feature value = 20.0
- URLCharProb: SHAP contribution = 0.2783, Feature value = 0.061933179
- LetterRatioInURL: SHAP contribution = -0.2227, Feature value = 0.581
- SpacialCharRatioInURL: SHAP contribution = 0.1980, Feature value = 0.032
- NoOfExternalRef: SHAP contribution = 0.1964, Feature value = 124.0

Explain the model's decision in simple and clear language.

Requirements:
- Do not make

In [29]:
import os
from dotenv import load_dotenv

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

print("Gemini API Key Loaded:", bool(GEMINI_API_KEY))
print("OpenRouter API Key Loaded:", bool(OPENROUTER_API_KEY))

Gemini API Key Loaded: True
OpenRouter API Key Loaded: True


## Initialize LLM Clients

Google Gemini and OpenRouter clients are initialized using their respective API keys.

OpenRouter provides an OpenAI-compatible API interface, allowing the existing Python OpenAI client to communicate with OpenRouter by changing the API base URL.

In [30]:
from google import genai
from openai import OpenAI

# Gemini Client
gemini_client = genai.Client(
    api_key=GEMINI_API_KEY
)

# OpenRouter Client
openrouter_client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

print("Gemini and OpenRouter clients initialized successfully!")

Gemini and OpenRouter clients initialized successfully!


## OpenRouter API Test

A simple test request is sent to OpenRouter to verify that the API key and client configuration are working correctly before connecting the service to the SHAP explanation pipeline.

In [31]:
openrouter_response = openrouter_client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": "Explain phishing websites in one simple sentence."
        }
    ],
    temperature=0.3
)

print("OpenRouter Response:")
print(openrouter_response.choices[0].message.content)

OpenRouter Response:
Phishing websites are fake sites that trick you into giving away your passwords or personal information by pretending to be a real, trusted site (like your bank or email).


# Gemini Response 

In [32]:
from google.genai import types

gemini_response = gemini_client.models.generate_content(
    model="gemini-3.5-flash",
    contents=prompt,
    config=types.GenerateContentConfig(
        temperature=0.3
    )
)

gemini_text = gemini_response.text

print("===== GEMINI EXPLANATION =====")
print(gemini_text)

===== GEMINI EXPLANATION =====
Hello! I am your PhishGuard AI cybersecurity explanation assistant. 

Based on the analysis performed by our machine learning model, the submitted URL has been classified as **Phishing** with a **100.00% probability**. 

To help you understand why the machine learning model reached this definitive conclusion, we look at the key features that influenced its decision. Here is a simple, clear breakdown of the most important factors:

### 1. Extremely High Similarity to Known Brands (URL Similarity Index)
* **What the model saw:** A similarity index of 100.0.
* **Why it matters:** This was by far the most influential factor in the model's decision. It indicates that the URL is designed to look almost identical to a highly trusted, legitimate brand or website. This is a classic "look-alike" (typosquatting) tactic used by scammers to trick users into thinking they are on a real website (e.g., mimicking a bank or social media login page).

### 2. Excessive Self-

# OPEN Router Response

In [33]:
openrouter_response = openrouter_client.chat.completions.create(
    model="openrouter/free",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.3
)

openrouter_text = openrouter_response.choices[0].message.content

print("===== OPENROUTER EXPLANATION =====")
print(openrouter_text)

===== OPENROUTER EXPLANATION =====
## Model Decision Explanation

The machine learning model has classified this URL as **Phishing** with **100% probability**. Here is why, based on the most important features it considered:

### 🔴 Most Important Feature: URL Similarity
The **URLSimilarityIndex** contributed the **most** to the prediction, with a value of **100.0**. This means the URL in question is extremely similar to known phishing websites. The model sees this as the strongest signal that the website is a fake.

### 🟠 Other Strong Signals
The model also noticed several other suspicious patterns:

- **Many self-references (119):** The URL contains many internal links pointing back to itself, which is unusual and often a sign of a malicious site.
- **Many images (34):** A phishing site often loads many images to distract users or hide malicious content.
- **Many JavaScript files (28):** JavaScript is commonly used to execute harmful code on a website.
- **Many CSS files (20):** CSS f

# Side By Side Comparision 

In [34]:
comparison = {
    "Prediction": prediction,
    "Phishing Probability": probability,
    "Gemini Explanation": gemini_text,
    "OpenRouter Explanation": openrouter_text
}

print("=" * 70)
print("PHISHGUARD AI - DUAL LLM EXPLANATION")
print("=" * 70)

print("\nPrediction:")
print(prediction)

print("\nPhishing Probability:")
print(f"{probability:.2%}")

print("\n--- GEMINI ---")
print(gemini_text)

print("\n--- OPENROUTER ---")
print(openrouter_text)

PHISHGUARD AI - DUAL LLM EXPLANATION

Prediction:
Phishing

Phishing Probability:
100.00%

--- GEMINI ---
Hello! I am your PhishGuard AI cybersecurity explanation assistant. 

Based on the analysis performed by our machine learning model, the submitted URL has been classified as **Phishing** with a **100.00% probability**. 

To help you understand why the machine learning model reached this definitive conclusion, we look at the key features that influenced its decision. Here is a simple, clear breakdown of the most important factors:

### 1. Extremely High Similarity to Known Brands (URL Similarity Index)
* **What the model saw:** A similarity index of 100.0.
* **Why it matters:** This was by far the most influential factor in the model's decision. It indicates that the URL is designed to look almost identical to a highly trusted, legitimate brand or website. This is a classic "look-alike" (typosquatting) tactic used by scammers to trick users into thinking they are on a real website (

# Saving The output 

In [35]:
llm_results = {
    "prediction": prediction,
    "phishing_probability": probability,
    "gemini_explanation": gemini_text,
    "openrouter_explanation": openrouter_text
}

with open("../models/llm_results.json", "w") as f:
    json.dump(llm_results, f, indent=4)

print("LLM results saved successfully!")

LLM results saved successfully!


## Conclusion

The LLM explanation module was successfully integrated with the existing
XGBoost and SHAP pipeline.

A structured prompt was created using the XGBoost prediction, phishing
probability, and top SHAP feature contributions. The same evidence is
provided to both LLM providers to ensure a fair comparison of their
generated explanations.

The LLMs are explicitly instructed not to perform a new phishing
classification. Instead, they convert the existing machine learning
prediction and SHAP evidence into a simple, user-friendly explanation.

Google Gemini and OpenRouter were integrated as independent LLM providers,
allowing their explanation quality to be compared using the same input
evidence.

The complete explainable pipeline is therefore:

XGBoost → SHAP → Structured Evidence → LLM → Human-readable Explanation